# Token probabilities

[Back to the introduction](ch01_demo.ipynb). Run in order in a fresh kernel. This loads SmolLM2-135M-Instruct (about 140 MB, WASM) and streams its answer token by token, each followed by its probability in brackets. A second line then shows the runner-up at every step: the most probable token the model did *not* pick. See [Token probabilities](https://github.com/decitre/lmlite/blob/main/docs/USAGE.md#token-probabilities) for how probabilities are defined.

In [ ]:
from lmlite import LLM

In [ ]:
token_llm = await LLM.create(
    generator_model="HuggingFaceTB/SmolLM2-135M-Instruct",
    backend="wasm",
    max_new_tokens=60,
    do_sample=False,  # Greedy: each picked token is the most probable one.
)

`chat_tokens()` yields one record per token: its `text`, its `prob`, and the `alternatives` most probable other tokens at that step. With sampling off, the picked token always has the highest probability.

In [ ]:
question = [{"role": "user", "content": "Explain embeddings simply."}]
tokens = []
runner_ups = []
async for token in token_llm.chat_tokens(question, alternatives=1):
    print(f"{token['text']}[{token['prob']:.2f}]", end="", flush=True)
    tokens.append(token['text'])
    runner_ups.append(token["alternatives"][0])

The synthetic sentences below joins the runner-up from every step. It is not text the model would write: each runner-up was scored after the *picked* tokens before it, not after the previous runner-ups. A runner-up that is half of a multi-byte character shows as `�`.

In [ ]:
print("".join(f"{alt['text']}[{alt['prob']:.2f}]" for alt in runner_ups))

In [ ]:
print("".join(f"{token} [{alt['text'].strip()}]" for token, alt in zip(tokens, runner_ups)))

In [ ]:
await token_llm.dispose()

The model is disposed without clearing cached downloads. Rerun the notebook to load it again from the cache.